### extract indictment facts with API gpt-VERDICTS

In [ ]:
import pandas as pd
import os
import re
from openai import OpenAI
import gc
from tqdm import tqdm
import csv
import time


DRY_RUN = False  # Set to False to process files and call the GPT API
PROCESS_ONLY_TARGET_VERDICTS = True  # Set to True to process only verdicts from target.csv
domain="wep"


# ========== API Setup ==========
os.environ["OPENAI_API_KEY"] = "REPLACED_OPENAI_KEY"  
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ========== File Paths ==========

if domain=="drugs":
    base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/"
else:
    base_path="/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/"
csv_directory =base_path+'verdict_csv'
out_dir = base_path+"gpt"

# base_path='/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/data/drugs_3k/docx'
# csv_directory ='/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/data/drugs_3k/verdict_csv'
# out_dir = '/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/data/drugs_3k'

os.makedirs(out_dir, exist_ok=True)

output_file = os.path.join(out_dir, "processed_verdicts_with_gpt.csv")
failed_file = os.path.join(out_dir, "failed_verdicts.csv")

# ========== Load Target CSV and Get Unique Verdicts (if flag is set) ==========
unique_verdicts = None
if PROCESS_ONLY_TARGET_VERDICTS:
    target_csv_path = os.path.join(base_path, "target.csv")
    if os.path.exists(target_csv_path):
        target_df = pd.read_csv(target_csv_path)
        # Extract unique verdicts from both verdict_1 and verdict_2 columns
        unique_verdicts = set(target_df['verdict_1'].astype(str).str.strip().unique()) | set(target_df['verdict_2'].astype(str).str.strip().unique())
        print(f"✅ Processing only {len(unique_verdicts)} unique verdicts from target CSV")
    else:
        print(f"⚠️ Target CSV not found at {target_csv_path}, processing all verdicts")
else:
    print(f"ℹ️ Processing all verdicts (PROCESS_ONLY_TARGET_VERDICTS is False)")

# ========== Pattern Definitions ==========
START_PARTS = ["עובדותם", "כללי", "כתב האישום", "האישום", "אישום", "רקע", "גזר", "דין", "פסק","מבוא","הרשעת" ,"בעניינו","עבירות","הורשע","עובדות","השתלשלות", "ג ז ר",  "ד י ן","פתח דבר","פתח"]
END_PARTS = ["טענות", "עמדת", "תסקיר","תסקירי", "שירות", "מבחן", "דיון", "התסקיר","טיעוני", "הצדדים", "צדדים", "והכרעה",  "ראיות","החלטה"]

# ========== Helper Functions ==========
def extract_indictment_facts(df):
    if df.empty or "part" not in df.columns or "text" not in df.columns:
        return "❌ No indictment facts found", None, None, 0

    df["part"] = df["part"].astype(str).str.strip()
    
    # Exclude irrelevant parts from being considered as start parts
    EXCLUDED_START_PARTS = ["כתבי עת", "חקיקה שאוזכרה", "חקיקה", "ציטוטים", "מקורות"]
    
    start_row = df[df["part"].str.contains('|'.join(START_PARTS), case=False, na=False, regex=True)]
    # Filter out excluded parts
    if not start_row.empty:
        excluded_mask = start_row["part"].str.contains('|'.join(EXCLUDED_START_PARTS), case=False, na=False, regex=True)
        start_row = start_row[~excluded_mask]
    
    if start_row.empty:
        # If no valid start part found, try to find any part that contains indictment keywords in its text
        # This is a fallback for cases where the part name doesn't match START_PARTS but the content does
        for idx, row in df.iterrows():
            text_content = str(row.get("text", "")).strip() if pd.notna(row.get("text")) else ""
            if text_content:
                text_lower = text_content.casefold()
                # Check if this text contains indictment keywords
                indictment_keywords = ["הורשע", "הרשענו", "מצאנו להרשיעו", "כתב אישום", "הנאשם הורשע"]
                if any(keyword in text_lower for keyword in indictment_keywords):
                    start_idx = idx
                    start_part_name = df.loc[idx, "part"]
                    normalized_start_part = re.sub(r"\s+", " ", str(start_part_name).strip().casefold())
                    has_start = True
                    break
        else:
            # No valid start found even after fallback search
            start_idx = 0
            start_part_name = "❌ No start found (use index 0)"
            normalized_start_part = None
            has_start = False
    else:
        start_idx = start_row.index.min()
        start_part_name = df.loc[start_idx, "part"]
        normalized_start_part = re.sub(r"\s+", " ", str(start_part_name).strip().casefold())
        has_start = True

    end_mask = (
        (df.index > start_idx) &
        (df["part"].str.contains('|'.join(END_PARTS), case=False, na=False, regex=True))
    )
    end_row = df.loc[end_mask]

    end_candidates = end_row.index.tolist()
    valid_end_idx = None

    for candidate_idx in end_candidates:
        candidate_part = str(df.loc[candidate_idx, "part"]).strip()
        if has_start:
            normalized_candidate = re.sub(r"\s+", " ", candidate_part.casefold())
            if normalized_candidate == normalized_start_part:
                continue
            # If end part includes the start part, skip it
            if normalized_start_part and normalized_start_part in normalized_candidate:
                continue
        # Skip end candidates that also match START_PARTS patterns
        # Any part that matches START_PARTS cannot be an end part (even if excluded from being a start part)
        # Use the same regex pattern matching as used for finding start parts
        if candidate_part:
            # Create a Series with the candidate part to use str.contains (same as start detection)
            candidate_part_lower = candidate_part.casefold()
            # Check each START_PARTS keyword individually to ensure proper matching
            matches_start_pattern = False
            for start_keyword in START_PARTS:
                # Check if the keyword appears in the candidate part
                if start_keyword in candidate_part_lower:
                    matches_start_pattern = True
            
            if matches_start_pattern:
                # If it matches START_PARTS, it cannot be an end part - skip it
                # (This handles cases like "גזר דין...דיון" which shouldn't be an end part)
                continue  # Skip any part that matches START_PARTS
                    break
        valid_end_idx = candidate_idx
            
            if matches_start_pattern:
                # If it matches START_PARTS, it cannot be an end part - skip it
                # (This handles cases like "גזר דין...דיון" which shouldn't be an end part)
                continue  # Skip any part that matches START_PARTS
        break

    if valid_end_idx is not None:
        end_idx = valid_end_idx
        end_part_name = df.loc[end_idx, "part"]
    else:
        end_idx = len(df)
        end_part_name = "❌ No end found (used full text)"

    # Count parts between start and end (inclusive of start, exclusive of end)
    parts_count = end_idx - start_idx
    
    # Extract text grouped by part with part names as headers
    extracted_sections = []
    current_part = None
    current_text_parts = []
    
    for idx in range(start_idx, end_idx):
        row = df.loc[idx]
        part_name = str(row["part"]).strip()
        text_content = str(row["text"]).strip() if pd.notna(row["text"]) else ""
        
        if not text_content:
            continue
            
        # If this is a new part, save the previous part and start a new one
        if part_name != current_part:
            if current_part is not None and current_text_parts:
                # Add the previous part with its header
                extracted_sections.append(f"{current_part}:")
                extracted_sections.append("\n".join(current_text_parts))
                extracted_sections.append("")  # Empty line between parts
            
            current_part = part_name
            current_text_parts = [text_content]
        else:
            # Same part, just append the text
            current_text_parts.append(text_content)
    
    # Don't forget the last part
    if current_part is not None and current_text_parts:
        extracted_sections.append(f"{current_part}:")
        extracted_sections.append("\n".join(current_text_parts))
    
    extracted_text = "\n".join(extracted_sections)
    
    # Validate that the extracted text actually contains indictment-related content
    # If it only contains citations, legislation, or other non-indictment content, return empty
    if extracted_text:
        extracted_text_lower = extracted_text.casefold()
        # Keywords that indicate this is actually an indictment section
        indictment_keywords = [
            "הורשע", "הרשענו", "מצאנו להרשיעו", "כתב אישום", "כתב האישום",
            "על פי הודאתו", "על פי הודאת", "הודה", "הנאשם הורשע", "הנאשם הודה",
            "על פי הנטען בכתב האישום", "על פי עובדות הכרעת הדין", "על פי עובדות כתב האישום",
            "על פי הממצאים שנקבעו בהכרעת הדין", "בכתב האישום", "בכתב אישום",
            "הסדר טיעון", "בעבירות", "לפי סעיף", "לפי סעיפים"
        ]
        
        # Check if text contains any indictment keywords
        has_indictment_content = any(keyword in extracted_text_lower for keyword in indictment_keywords)
        
        # If the text is mostly citations/legislation (contains many law references but no indictment)
        # or if it's very short and doesn't contain indictment keywords, it's probably not an indictment
        if not has_indictment_content:
            # Check if it's mostly citations/legislation
            citation_indicators = ["חוק העונשין", "פקודת", "תקנות", "ע\"פ", "ע\"א", "ע\"מ", "ע\"ב"]
            has_citations = any(indicator in extracted_text_lower for indicator in citation_indicators)
            
            # If it has citations but no indictment content, it's probably not an indictment section
            if has_citations and len(extracted_text.split()) < 50:  # Short text with citations but no indictment
                return "❌ No indictment facts found", start_part_name, end_part_name, parts_count
    
    return extracted_text.strip() if extracted_text else "❌ No indictment facts found", start_part_name, end_part_name, parts_count


def extract_facts_with_gpt(text):
    """
    Sends extracted text to GPT API and extracts specific facts.
    """
    if text == "❌ No indictment facts found" :
        return "GPT extraction error"


    prompt = f"""
תפקידך הוא לחלץ מידע משפטי מתוך טקסט של גזר דין.
המטרה שלך היא למצוא את "הסיפור העובדתי" - בגין מה הורשע הנאשם ומה בדיוק קרה שם.

עליך לחלץ שני חלקים:
1. **פסקת האישום/ההרשעה**: המשפט הפורמלי שקובע במה הנאשם הורשע (סעיפי חוק, סוג העבירה, הודאה/הכחשה).
2. **תיאור העובדות**: הסיפור המלא של המקרה (מה קרה, מתי, איפה, מי המעורבים).

הנחיות לביצוע:
1. חפש עוגנים כמו: "הנאשם הורשע", "על פי עובדות כתב האישום", "כתב האישום המתוקן", "העובדות בהן הודה".
2. אם הטקסט מכיל תיאור עובדתי (סיפור המעשה) מיד לאחר ההרשעה - העתק את כולו.
3. **אל תסכם**. העתק את הטקסט המקורי מילה במילה (Copy-Paste).
4. **מתי לעצור?** הפסק להעתיק כאשר הטקסט עובר לנושאים אחרים כגון: "תסקיר שירות המבחן", "טיעונים לעונש", "ראיות לעונש", "דיון והכרעה" או ניתוח משפטי.


כעת, עבד את הטקסט הבא:
  {text}

    החזר את הפלט בפורמט הבא בלבד:
    <פסקת כתב האישום>

    <פסקת עובדות כתב האישום>


    """

    response = client.chat.completions.create(
        model="gpt-4.1", 
        messages=[
            {"role": "system", "content": "אתה מודל בינה מלאכותית שתפקידו לחלץ עובדות מכתבי אישום בטקסטים משפטיים בעברית, מבלי לפרש, לסכם או לשנות את הנוסח המקורי."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()
     
# בדיקה האם יש פסקת כתב אישום
def is_valid_indictment(text):
    # Handle NaN, None, or empty values
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return False
    if len(text.strip().split("\n")) < 1:
        return False
    first_paragraph = text.strip().split("\n")[0]
    keywords = [
        "הנאשם הורשע","הנאשם הודה","הנאשם הורשע על פי הודאתו","על פי הודאתו","על פי הודאת הנאשם","הודה בעובדות כתב האישום","הורשע בעובדות כתב האישום","כתב אישום","בעבירות לפי סעיף","במסגרת הסדר טיעון","בגזר הדין","הורשע על פי הסדר",
        "הנאשם הובא לדין","נגד הנאשם הוגש כתב אישום", "הורשע במסגרת","הורשע בעבירות של","בהתאם לכתב האישום","מכתב האישום עולה כי", "הנאשם יוחסו עבירות של","הורשע לאחר שהודה",'הודייתו',"הורשע","הודיתו","בכתב האישום","הודה","כפר","אישום","כתב האישום","בהתאם לעובדות"
    ]
    return any(kw in first_paragraph for kw in keywords)



# ========== Load Existing Data ==========
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    # Create backup for comparison (only if backup doesn't exist)
    backup_file = output_file.replace(".csv", "_old.csv")
    if not os.path.exists(backup_file):
        import shutil
        shutil.copy2(output_file, backup_file)
        print(f"📋 Created backup of existing results: {backup_file}")
        print(f"   (This backup will be used for comparison after processing)")
else:
    processed_df = pd.DataFrame(columns=["verdict", "extracted_facts", "extracted_gpt_facts","start_part","end_part", "parts_count"])

# ========== Process Files ==========
file_list = [f for f in os.listdir(csv_directory) if f.endswith(".csv")]
processed_df["verdict"] = processed_df["verdict"].astype(str).str.strip()
failed_verdicts = []
unique_start_end_pairs = set()
parts_count_stats = []  # Track parts count for each verdict

for filename in tqdm(file_list, desc="Processing verdicts"):
    file_path = os.path.join(csv_directory, filename)
    try:
        df = pd.read_csv(file_path)
        verdict_id = str(df["verdict"].iloc[0]).strip()
        
        # ========== Skip if verdict is not in target CSV ==========
        if unique_verdicts is not None and verdict_id not in unique_verdicts:
            continue

        # Extract
        extracted_facts, start_part, end_part, parts_count = extract_indictment_facts(df)
        extracted_facts_normalized = extracted_facts.strip() if isinstance(extracted_facts, str) else ""
        start_label = start_part.strip() if isinstance(start_part, str) else str(start_part)
        end_label = end_part.strip() if isinstance(end_part, str) else str(end_part)
        unique_start_end_pairs.add((start_label, end_label))
        
        # Track parts count statistics
        parts_count_stats.append({
            "verdict": verdict_id,
            "parts_count": parts_count,
            "start_part": start_label,
            "end_part": end_label
        })

        # Check if already processed and if extraction is valid (not empty and not too short)
        # Check if GPT extraction is valid (not empty and not too short)
        def is_valid_extraction(gpt_text):
            if pd.isna(gpt_text) or not isinstance(gpt_text, str):
                return False
            gpt_text = gpt_text.strip()
            if gpt_text == "" or gpt_text == "GPT extraction error":
                return False
            # Check if it's too short (≤2 sentences)
            sentences = [s.strip() for s in re.split(r'[.!?]', gpt_text) if s.strip()]
            if len(sentences) <= 2:
                return False
            return True

        if DRY_RUN:
            continue

        # Check if already processed and if extraction is valid
        existing_rows = processed_df[processed_df["verdict"] == verdict_id]
        if not existing_rows.empty:
            existing_gpt = existing_rows["extracted_gpt_facts"].iloc[0]
            # If extraction is valid, skip GPT call and use existing result
            if is_valid_extraction(existing_gpt):
                print(f"⏭️ Skipping GPT for verdict {verdict_id} (valid extraction exists)")
                continue

        # Only run GPT extraction if no valid extraction exists
        extracted_gpt_facts = extract_facts_with_gpt(extracted_facts)
        
        # if not is_valid_indictment(extracted_gpt_facts):
        #     failed_verdicts.append({
        #         "verdict": verdict_id,
        #         "reason": "Invalid or missing indictment",
        #         "first_chars": extracted_facts[:300]
        #     })
        #     continue

        # Save or update
        existing_rows = processed_df[processed_df["verdict"] == verdict_id]
        if not existing_rows.empty:
            # Update existing row
            idx = existing_rows.index[0]
            processed_df.at[idx, "extracted_facts"] = extracted_facts
            processed_df.at[idx, "extracted_gpt_facts"] = extracted_gpt_facts
            processed_df.at[idx, "start_part"] = start_part
            processed_df.at[idx, "end_part"] = end_part
            processed_df.at[idx, "parts_count"] = parts_count
        else:
            # Create new row
            new_row = pd.DataFrame([{
                "verdict": verdict_id,
                "extracted_facts": extracted_facts,
                "extracted_gpt_facts": extracted_gpt_facts,
                "start_part": start_part,
                "end_part": end_part,
                "parts_count": parts_count
            }])
            processed_df = pd.concat([processed_df, new_row], ignore_index=True)
        processed_df.to_csv(output_file, index=False, encoding="utf-8-sig")
        
        time.sleep(1)  # avoid rate limits

    except Exception as e:
        failed_verdicts.append({"verdict": filename, "reason": str(e)})

    gc.collect()

if DRY_RUN:
    print(f"Found {len(unique_start_end_pairs)} unique start/end combinations.")
    for start_label, end_label in sorted(unique_start_end_pairs, key=lambda pair: (pair[0], pair[1])):
        print(f"START: {start_label} | END: {end_label}")

# ========== Save Failures ==========
if failed_verdicts:
    pd.DataFrame(failed_verdicts).to_csv(failed_file, index=False, encoding="utf-8-sig")

# ========== Empty and Short Extractions Analysis ==========
print("\n" + "="*60)
print("EMPTY AND SHORT EXTRACTIONS ANALYSIS")
print("="*60)

if not processed_df.empty and "extracted_gpt_facts" in processed_df.columns and "extracted_facts" in processed_df.columns:
    # Find empty extractions
    def is_empty(text):
        if pd.isna(text) or not isinstance(text, str):
            return True
        return text.strip() == "" or text.strip() == "GPT extraction error"
    
    # Find short extractions (up to 2 sentences)
    def is_short(text):
        if pd.isna(text) or not isinstance(text, str) or is_empty(text):
            return False
        # Count sentences (split by period, exclamation, question mark)
        sentences = [s.strip() for s in re.split(r'[.!?]', text.strip()) if s.strip()]
        return len(sentences) <= 2
    
    processed_df["is_empty"] = processed_df["extracted_gpt_facts"].apply(is_empty)
    processed_df["is_short"] = processed_df["extracted_gpt_facts"].apply(is_short)
    
    empty_extractions = processed_df[processed_df["is_empty"]].copy()
    short_extractions = processed_df[processed_df["is_short"] & ~processed_df["is_empty"]].copy()
    
    empty_count = len(empty_extractions)
    short_count = len(short_extractions)
    
    print(f"\n📊 Empty Extractions: {empty_count} out of {len(processed_df)} total")
    print(f"📊 Short Extractions (≤2 sentences): {short_count} out of {len(processed_df)} total")
    
    all_problematic = pd.concat([empty_extractions, short_extractions]).drop_duplicates(subset=['verdict'])
    
    if len(all_problematic) > 0:
        print(f"\n📝 Input Text for Empty/Short Extractions:")
        for idx, row in all_problematic.iterrows():
            verdict_id = row['verdict']
            input_text = str(row['extracted_facts']) if pd.notna(row['extracted_facts']) else ""
            gpt_output = str(row['extracted_gpt_facts']) if pd.notna(row['extracted_gpt_facts']) else ""
            
            is_empty_case = row['is_empty']
            is_short_case = row['is_short'] if not is_empty_case else False
            
            # Show which parts were included
            if "start_part" in row and "end_part" in row:
                start_part = row['start_part']
                end_part = row['end_part']
                parts_count = row.get('parts_count', 'N/A')
                print(f"\n  Verdict: {verdict_id}")
                print(f"  Start part: {start_part}")
                print(f"  End part: {end_part}")
                print(f"  Parts included: {parts_count}")
                if is_empty_case:
                    print(f"  Status: EMPTY")
                elif is_short_case:
                    print(f"  Status: SHORT (≤2 sentences)")
            
            # Show part names from the input text (lines ending with :)
            part_names = [line.strip() for line in input_text.split("\n") if line.strip().endswith(":")]
            if part_names:
                print(f"  Part names in input: {', '.join(part_names[:5])}")
            
            if gpt_output:
                print(f"  GPT output: {gpt_output}")
            else:
                print(f"  GPT output: (empty)")
            
            print(f"  Input text: {input_text}")
    else:
        print("\n✅ No empty or short extractions found!")
else:
    print("⚠️ No processed data available for analysis.")



✅ Processing only 101 unique verdicts from target CSV


Processing verdicts:   5%|▍         | 75/1640 [00:36<14:22,  1.81it/s]

## update similarity_database_with_indicment_facts with new extracted facts

In [6]:
import pandas as pd
import os

def update_similarity_database(
    similarity_db_path: str,
    new_facts_csv_path: str,
    output_path: str,
    # --- Column Configuration (Change these if your CSV headers are different) ---
    sim_verdict1_col: str = 'verdict_1',        # ID column in Similarity DB
    sim_verdict2_col: str = 'verdict_2',        # ID column in Similarity DB
    sim_facts1_col: str = 'indicment_facts_1',  # Target text col in Similarity DB
    sim_facts2_col: str = 'indicment_facts_2',  # Target text col in Similarity DB
    
    new_id_col: str = 'verdict',             # ID column in New Facts CSV
    new_text_col: str = 'extracted_gpt_facts'      # Text column in New Facts CSV
):
    """
    Updates the indictment facts in the similarity database using a master facts CSV.
    """
    print(f"📂 Loading Similarity DB: {similarity_db_path}")
    if not os.path.exists(similarity_db_path):
        print(f"❌ Error: File not found - {similarity_db_path}")
        return
    df_sim = pd.read_csv(similarity_db_path)
    
    print(f"📂 Loading New Facts CSV: {new_facts_csv_path}")
    if not os.path.exists(new_facts_csv_path):
        print(f"❌ Error: File not found - {new_facts_csv_path}")
        return
    df_facts = pd.read_csv(new_facts_csv_path)
    
    # 1. Create Lookup Dictionary (ID -> Fact)
    # Ensure IDs are strings and stripped of whitespace for accurate matching
    print("⚙️  Creating lookup map...")
    df_facts[new_id_col] = df_facts[new_id_col].astype(str).str.strip()
    
    # Remove duplicates in master file (keep last or first valid fact)
    df_facts = df_facts.drop_duplicates(subset=[new_id_col], keep='last')
    
    # Convert to dictionary
    facts_map = pd.Series(
        df_facts[new_text_col].values, 
        index=df_facts[new_id_col]
    ).to_dict()
    
    print(f"   Mapped {len(facts_map)} unique verdicts from master file.")

    # 2. Update Similarity DataFrame
    print("⚙️  Updating rows in Similarity Database...")
    
    # Convert similarity IDs to string for matching
    df_sim[sim_verdict1_col] = df_sim[sim_verdict1_col].astype(str).str.strip()
    df_sim[sim_verdict2_col] = df_sim[sim_verdict2_col].astype(str).str.strip()
    
    # Map new values
    # If a verdict ID is NOT found in the new file, we keep the old value (fillna)
    
    # Update facts_1
    new_values_1 = df_sim[sim_verdict1_col].map(facts_map)
    if sim_facts1_col in df_sim.columns:
        original_1 = df_sim[sim_facts1_col]
        df_sim[sim_facts1_col] = new_values_1.fillna(original_1)
    else:
        df_sim[sim_facts1_col] = new_values_1.fillna("")

    # Update facts_2
    new_values_2 = df_sim[sim_verdict2_col].map(facts_map)
    if sim_facts2_col in df_sim.columns:
        original_2 = df_sim[sim_facts2_col]
        df_sim[sim_facts2_col] = new_values_2.fillna(original_2)
    else:
        df_sim[sim_facts2_col] = new_values_2.fillna("")

    # 3. Save
    print(f"💾 Saving updated file to: {output_path}")
    df_sim.to_csv(output_path, index=False)
    print("✅ Done!")

# ==========================================
# RUN CONFIGURATION
# ==========================================
if __name__ == "__main__":
    # 1. Path to your existing Similarity Database (the pairs file)
    SIM_DB_PATH = base_path+"similarity_database_with_indicment_facts.csv"
    
    # 2. Path to the NEW CSV with the correct facts (the master list)
    NEW_FACTS_PATH = base_path+'gpt_only_similarity/processed_verdicts_with_gpt.csv'
    
    # 3. Output file name
    OUTPUT_PATH = base_path+"similarity_database_with_indicment_facts.csv"    
    # 4. Column Name Mapping (Check your CSV headers!)
    # What are the column names in the NEW file?
    NEW_ID_COLUMN = "verdict"          # e.g., 'id', 'verdict_id', 'CaseID'
    NEW_TEXT_COLUMN = "extracted_gpt_facts"  # e.g., 'facts', 'text', 'indictment_facts'

    update_similarity_database(
        similarity_db_path=SIM_DB_PATH,
        new_facts_csv_path=NEW_FACTS_PATH,
        output_path=OUTPUT_PATH,
        new_id_col=NEW_ID_COLUMN,
        new_text_col=NEW_TEXT_COLUMN
    )

📂 Loading Similarity DB: /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/similarity_database_with_indicment_facts.csv
📂 Loading New Facts CSV: /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/gpt_only_similarity/processed_verdicts_with_gpt.csv
⚙️  Creating lookup map...
   Mapped 104 unique verdicts from master file.
⚙️  Updating rows in Similarity Database...
💾 Saving updated file to: /Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/weapon/similarity_database_with_indicment_facts.csv
✅ Done!


## GPT verification

In [1]:
import pandas as pd
import os
import json
from openai import OpenAI
import gc
from tqdm import tqdm
import time
os.environ["OPENAI_API_KEY"] = ""  

class GPTVerifier:
    def __init__(self, verification_model="gpt-4.1"):
        # Only OpenAI client needed
        self.openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.verification_model = verification_model
        print(f"Using {verification_model} for verification")
    
    def verify_with_different_gpt(self, original_text, gpt_extraction):
        """Verify extraction using a different GPT model for cross-validation"""
        prompt = f"""
אתה מערכת ביקורת לחילוץ עובדות משפטיות מכתבי אישום. 
המטרה שלך היא לבדוק אם החילוץ שמתקבל ממערכת אחרת אחר מדויק, שלם, ורלוונטי.


הטקסט המקורי:
{original_text}

החילוץ:
{gpt_extraction}

אנא דרג את החילוץ על פי הקריטריונים הבאים:
1. האם החילוץ כולל את פסקת כתב האישום? (true/false)
2. עד כמה מדויק המידע שהופיע בחילוץ? (0–10)
3. עד כמה החילוץ שלם – האם חסר מידע משמעותי? (0–10)
4. האם יש מידע שאינו רלוונטי לחלק של העובדות? (0–10, ציון גבוה = רלוונטיות טובה)

חוות דעת:
- תן הערות ספציפיות על בעיות בחילוץ, אם יש.

החזר תשובה בפורמט JSON הבא (בלי הסברים נוספים):
{{
  "is_valid": true/false,
  "has_indictment": true/false,
  "accuracy_score": int,
  "completeness_score": int,
  "relevance_score": int,
  "issues": ["problem1", "problem2"],
  "feedback": "..."
}}
"""
        try:
            response = self.openai_client.chat.completions.create(
                model=self.verification_model,
                messages=[
                    {"role": "system", "content": "אתה בודק איכות משפטי מקצועי. תפקידך להעריך את איכות החילוץ של עובדות מכתבי אישום, על פי קריטריונים ברורים. עליך לספק תשובה אובייקטיבית, מדויקת, תמציתית ובפורמט JSON בלבד. אל תחרוג מהפורמט."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0  # For consistency
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            return {"error": str(e)}
    
    def rule_based_verification(self, original_text, gpt_extraction):
        """Rule-based verification using patterns"""
        issues = []
        
        # Check for indictment keywords
        indictment_keywords = [
            "הורשע", "הודה", "הודאתו", "כתב האישום", "לפי סעיף",
            "כתב אישום", "בעבירות", "בעבירה", "הודאתו", "הודאת הנאשם", "הודה בעובדות כתב האישום",
            "הורשע בעובדות כתב האישום", "בעבירות לפי סעיף", "במסגרת הסדר טיעון", "בגזר הדין", "הורשע על פי הסדר",
            "הנאשם הובא לדין", "נגד הנאשם הוגש כתב אישום", "הורשע במסגרת", "הורשע בעבירות של", "בהתאם לכתב האישום",
            "מכתב האישום עולה כי", "הנאשם יוחסו עבירות של", "הורשע לאחר שהודה", "הודייתו", "הודיתו", "בכתב האישום",
            "הודה", "כפר", "אישום", "הורשעו", "הורשעה", "הודתה", "הודו", "הוגש כתב אישום", "הוגש נגדו כתב אישום",
            "הוגש נגדה כתב אישום", "הוגשו נגדו כתבי אישום", "הוגשו נגדה כתבי אישום", "הורשעו בעבירות", "הורשעה בעבירות",
            "הודו בעבירות", "הודתה בעבירות", "הודו בעבירה", "הודתה בעבירה", "הורשעו בעבירה", "הורשעה בעבירה"
        ]
        
        
        has_indictment = any(kw in gpt_extraction for kw in indictment_keywords)
        if not has_indictment:
            issues.append("לא נמצאה פסקת כתב אישום")
        
        # Check length ratio
        extraction_length = len(gpt_extraction.split())
        original_length = len(original_text.split())
        
        if extraction_length > original_length * 0.9:
            issues.append("החילוץ ארוך מדי יחסית לטקסט המקורי")
        elif extraction_length < 20:
            issues.append("החילוץ קצר מדי")
        
        
        return {
            "has_indictment": has_indictment,
            "issues": issues,
            "length_ratio": extraction_length / original_length if original_length > 0 else 0,
            "rule_score": max(0, 10 - len(issues) * 2)  # Simple scoring
        }
    
    def comprehensive_verify(self, original_text, gpt_extraction):
        """Run verification using different GPT model + rules"""
        results = {}
        
        # GPT verification
        results["gpt_verification"] = self.verify_with_different_gpt(original_text, gpt_extraction)
        
        # Rule-based verification
        results["rule_based"] = self.rule_based_verification(original_text, gpt_extraction)
        
        # Calculate average score
        gpt_score = results["gpt_verification"].get("accuracy_score", 0)
        rule_score = results["rule_based"].get("rule_score", 0)
        
        results["average_score"] = (gpt_score + rule_score) / 2
        results["is_valid"] = (
            results["gpt_verification"].get("is_valid", False) and 
            results["rule_based"].get("has_indictment", False)
        )
        
        return results


def run_verification(input_csv, output_csv, verification_model="gpt-4o"):
    """Main function to run verification on extracted facts"""
    verifier = GPTVerifier(verification_model=verification_model)
    
    # Load existing extractions
    df = pd.read_csv(input_csv)
    
    # Prepare output columns
    verification_columns = [
        'verification_score', 'rule_score', 'average_score',
        'is_valid', 'issues', 'gpt_issues', 'gpt_feedback',
        'verification_status', 'verification_model'
    ]
    
    for col in verification_columns:
        if col not in df.columns:
            df[col] = None
    
    # Process each row
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Verifying with {verification_model}"):
        if pd.notna(row.get('verification_status')):
            continue  # Skip already verified
        
        try:
            # Run verification
            verification_results = verifier.comprehensive_verify(
                row['extracted_facts'], 
                row['extracted_gpt_facts']
            )
            
            # Update DataFrame
            df.at[idx, 'verification_score'] = verification_results.get("gpt_verification", {}).get("accuracy_score", 0)
            df.at[idx, 'rule_score'] = verification_results.get("rule_based", {}).get("rule_score", 0)
            df.at[idx, 'average_score'] = verification_results.get("average_score", 0)
            df.at[idx, 'is_valid'] = verification_results.get("is_valid", False)
            df.at[idx, 'issues'] = json.dumps(verification_results.get("rule_based", {}).get("issues", []), ensure_ascii=False)
            df.at[idx, 'verification_status'] = 'completed'
            df.at[idx, 'verification_model'] = verification_model
            df.at[idx, 'gpt_issues'] = json.dumps(
            verification_results.get("gpt_verification", {}).get("issues", []), 
            ensure_ascii=False)
            df.at[idx, 'gpt_feedback'] = verification_results.get("gpt_verification", {}).get("feedback", "")

            # Save progress every 10 rows
            if idx % 10 == 0:
                df.to_csv(output_csv, index=False, encoding="utf-8-sig")
            
            time.sleep(1)  # Rate limiting
            
        except Exception as e:
            df.at[idx, 'verification_status'] = f'error: {str(e)}'
            print(f"Error processing row {idx}: {e}")
    
    # Final save
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    
    # Generate summary report
    generate_verification_report(df, output_csv.replace('.csv', '_report.txt'))


def generate_verification_report(df, report_path):
    """Generate a summary report of verification results"""
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=== GPT Verification Report ===\n\n")
        
        total_verified = len(df[df['verification_status'] == 'completed'])
        f.write(f"Total extractions verified: {total_verified}\n")
        
        if total_verified > 0:
            avg_verification_score = df['verification_score'].mean()
            avg_rule_score = df['rule_score'].mean()
            avg_overall_score = df['average_score'].mean()
            valid_extractions = len(df[df['is_valid'] == True])
            
            f.write(f"Average verification score: {avg_verification_score:.2f}\n")
            f.write(f"Average rule-based score: {avg_rule_score:.2f}\n")
            f.write(f"Average overall score: {avg_overall_score:.2f}\n")
            f.write(f"Valid extractions: {valid_extractions}/{total_verified} ({valid_extractions/total_verified*100:.1f}%)\n\n")
            
            # Quality distribution
            high_quality = len(df[df['average_score'] >= 8])
            medium_quality = len(df[(df['average_score'] >= 5) & (df['average_score'] < 8)])
            low_quality = len(df[df['average_score'] < 5])
            
            f.write(f"High quality extractions (8-10): {high_quality}\n")
            f.write(f"Medium quality extractions (5-7): {medium_quality}\n")
            f.write(f"Low quality extractions (0-4): {low_quality}\n\n")
            
            # Common issues
            all_issues = []
            for issues_str in df['issues'].dropna():
                try:
                    issues = json.loads(issues_str)
                    all_issues.extend(issues)
                except:
                    continue
            
            from collections import Counter
            issue_counts = Counter(all_issues)
            f.write("Most common issues:\n")
            for issue, count in issue_counts.most_common(5):
                f.write(f"  {issue}: {count} times\n")


# Usage examples - MUCH SIMPLER!
if __name__ == "__main__":
    input_file = "/home/liorkob/M.Sc/thesis/data/drugs_3k/gpt/processed_verdicts_with_gpt.csv"
    
    run_verification(input_file, "verified_with_gpt_4-1.csv", "gpt-4.1")
    
    # Option 2: Premium verification with GPT-4o
    # run_verification(input_file, "verified_with_gpt4o.csv", "gpt-4o")
    
    # Option 3: Balanced with GPT-4 Turbo
    # run_verification(input_file, "verified_with_gpt4turbo.csv", "gpt-4-turbo")

Using gpt-4.1 for verification


Verifying with gpt-4.1:  12%|█▏        | 358/2987 [07:37<54:02,  1.23s/it]  

: 

### extract indictment facts with API gpt-APPEALS

In [ ]:
import pandas as pd
import os
import re
from openai import OpenAI
import gc
from tqdm import tqdm
import csv
import time

# ========== API Setup ==========
os.environ["OPENAI_API_KEY"] = ""  # Replace with actual key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ========== Pattern Definitions ==========
START_PARTS = ["עובדותם", "כללי", "כתב האישום", "האישום", "אישום", "רקע", "גזר", "דין", "פסק","מבוא","הרשעת" ,"בעניינו","עבירות","הורשע","עובדות","השתלשלות", "ג ז ר",  "ד י ן", "פ ס ק","העובדות","עובדות","השופט","שופט","השופטת","שופטת","הערעור","ערעור"]
END_PARTS = ["אני מסכים" ,"טענות", "עמדת", "תסקיר", "תסקירי","שירות", "מבחן", "דיון", "התסקיר","טיעוני", "הצדדים", "צדדים", "והכרעה", "ראיות","הכרעה"]

# ========== File Paths ==========
csv_directory = "/home/liorkob/M.Sc/thesis/data/5k/appeals_csv"
out_dir = "/home/liorkob/M.Sc/thesis/data/5k/gpt"
os.makedirs(out_dir, exist_ok=True)
output_file = os.path.join(out_dir, "processed_appeals_with_gpt_2.csv")
failed_file = os.path.join(out_dir, "failed_verdicts.csv")

# ========== Helper Functions ==========
def extract_indictment_facts(df):
    if df.empty or "part" not in df.columns or "text" not in df.columns:
        return "❌ No indictment facts found", None, None

    df["part"] = df["part"].astype(str).str.strip()

    start_row = df[df["part"].str.contains('|'.join(START_PARTS), case=False, na=False, regex=True)]

    df["word_count"] = df["text"].astype(str).str.split().apply(len)
    mask = df["part"].str.contains("חקיקה שאוזכרה", na=False) | (df["part"].str.len() < 3)
    df.loc[mask, "word_count"] = 0

    total_words = df["word_count"].sum()

    if start_row.empty:
        start_idx = 0
        start_part_name = "❌ No start found (used full text)"
    else:
        start_idx = start_row.index.min()

        words_until_start = df.loc[:start_idx, "word_count"].sum()

        if words_until_start > total_words / 3:
            start_idx = 0
            start_part_name = "❌ No start found (used full text)"
        else:
            start_part_name = df.loc[start_idx, "part"]

    # ===== Find end part =====
    end_row = df[
        (df.index > start_idx) &
        (df["part"].str.contains('|'.join(END_PARTS), case=False, na=False, regex=True))
    ]
    if not end_row.empty and end_row.index.min() == start_idx:
        end_row = df[
            (df.index > start_idx + 1) &
            (df["part"].str.contains('|'.join(END_PARTS), case=False, na=False, regex=True))
        ]

    end_idx = end_row.index.min() if not end_row.empty else len(df)
    end_part_name = df.loc[end_idx, "part"] if not end_row.empty else "❌ No end found (used full text)"

    extracted_text = "\n".join(df.loc[start_idx:end_idx - 1, "text"].dropna().astype(str))
    
    return extracted_text.strip() if extracted_text else "❌ No indictment facts found", start_part_name, end_part_name

def extract_facts_with_gpt(text):
    if text == "❌ No indictment facts found":
        return "GPT extraction error"

    prompt = f"""
הוראות: הטקסט שלפניך הוא מתוך פסק דין בערעור. עליך לחלץ ממנו את פסקת כתב האישום והעובדות **כפי שתוארו בפסק הדין המקורי** (לא את נימוקי הערעור).  
יש לחלץ את הטקסט **בדיוק כפי שהוא**, ללא ניסוח מחדש וללא סיכום.

 **החזרה שלך צריכה לכלול שתי פסקאות בלבד, ובסדר הבא**:
1. פסקת כתב האישום – הכוללת את סעיפי החוק, ההודאה, ההרשעה וכל פרט פורמלי.
2. פסקת העובדות – תיאור רצף האירועים: מי, מה, מתי, איפה, איך.

 הנחיות:
- אין להוסיף כותרות, אין לשנות ניסוחים, אין להוסיף פרשנות.
- כלול רק טקסט המתאר את האישום והעובדות המקוריות – כפי שמתואר בערעור.
- אל תכלול עדויות, טענות הערעור או תיאורי הכרעה משפטית.
- אם אין מידע על כתב האישום – אל תחזיר דבר.

טקסט המקור:
{text}

החזר את הפלט בפורמט הבא בלבד:
<פסקת כתב האישום>

<פסקת עובדות כתב האישום>
    """

    response = client.chat.completions.create(
        model="gpt-4.1-mini", 
        messages=[
            {"role": "system", "content": "You extract indictment and factual paragraphs from appellate decisions, copying the original wording without interpretation or summarization."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()

# ========== Load Existing Output ==========
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
else:
    processed_df = pd.DataFrame(columns=["verdict", "extracted_facts", "extracted_gpt_facts", "start_part", "end_part"])

processed_df["verdict"] = processed_df["verdict"].astype(str).str.strip()

# ========== Processing Loop ==========
total_files = 0
successful_extractions = 0
failed_extractions = 0
failed_verdicts = []
def is_english(text):
    return bool(re.fullmatch(r'[A-Za-z0-9\s.,;:!?\'\"()\[\]{}\-_=+*&^%$#@/\\]+', text))

file_list = [f for f in os.listdir(csv_directory) if f.endswith(".csv")]

for filename in tqdm(file_list, desc="Processing verdicts"):
    total_files += 1
    file_path = os.path.join(csv_directory, filename)

    try:
        df = pd.read_csv(file_path)
        verdict_id = str(df["verdict"].iloc[0]).strip()

        # # Check if already processed successfully
        existing = processed_df[processed_df["verdict"] == verdict_id]
        if not existing.empty and pd.notna(existing["extracted_gpt_facts"].iloc[0]) and existing["extracted_gpt_facts"].iloc[0].strip() != "":
            print(f"⏭️ Skipping GPT for verdict (already processed): {verdict_id}")
            continue

        print(f"📨 Calling GPT for verdict: {verdict_id}")
        extracted_facts, start_part, end_part = extract_indictment_facts(df)
        extracted_gpt_facts = extract_facts_with_gpt(extracted_facts)
        print(extracted_gpt_facts)
        time.sleep(1)

        failure_message = "לא נמצא טקסט המתאר את כתב האישום או העובדות כפי שתוארו בפסק הדין המקורי בטקסט שניתן."

        if extracted_facts.startswith("❌") or is_english(extracted_gpt_facts) or failure_message in extracted_gpt_facts:
            failed_extractions += 1
            failed_verdicts.append({"verdict": verdict_id, "all_parts": "; ".join(df["part"].dropna().astype(str))})
            print(f"❌ GPT failed for verdict: {verdict_id}")

            continue

        successful_extractions += 1
        processed_df = processed_df[processed_df["verdict"] != verdict_id]

        new_row = {
            "verdict": verdict_id,
            "extracted_facts": extracted_facts,
            "extracted_gpt_facts": extracted_gpt_facts,
            "start_part": start_part,
            "end_part": end_part
        }
#         new_row = {
#     "verdict": verdict_id,
#     "extracted_facts": existing["extracted_facts"].iloc[0] if not existing.empty else extracted_facts,
#     "extracted_gpt_facts": existing["extracted_gpt_facts"].iloc[0] if not existing.empty else "GPT extraction error",
#     "new_extracted_gpt_facts": extracted_gpt_facts,
#     "start_part": start_part,
#     "end_part": end_part
# }

        processed_df = pd.concat([processed_df, pd.DataFrame([new_row])], ignore_index=True)
        # ✅ Save immediately after each verdict
        processed_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    except Exception as e:
        print(f"💥 Error processing {filename}: {e}")

    del df
    gc.collect()

# ========== Final Save ==========
processed_df.to_csv(output_file, index=False, encoding="utf-8-sig")

failed_df = pd.DataFrame(failed_verdicts)
failed_df.to_csv(failed_file, index=False, encoding="utf-8-sig")

print("\n=== Statistics ===")
print(pd.DataFrame([{ "Total CSV Files Processed": total_files, "Successful Extractions": successful_extractions, "Failed Extractions": failed_extractions }]))
if not failed_df.empty:
    print("\n=== Sample of Failed Verdicts ===")
    print(failed_df.head())
print("\n✅ Process complete. Results saved.")

In [ ]:
import pandas as pd
import re
import os

def is_english(text):
    return bool(re.fullmatch(r'[A-Za-z0-9\s.,;:!?\'\"()\[\]{}\-_=+*&^%$#@/\\]+', text))

# Load processed data
df = pd.read_csv("/home/liorkob/M.Sc/thesis/data/5k/gpt/processed_verdicts_with_gpt_2.csv")

failure_message = "לא נמצא טקסט המתאר את כתב האישום או העובדות כפי שתוארו בפסק הדין המקורי בטקסט שניתן."

# Identify failures
failed_df = df[
    df["extracted_facts"].str.startswith("❌") |
    df["extracted_gpt_facts"].apply(is_english) |
    df["extracted_gpt_facts"].astype(str).str.contains(failure_message, na=False)
]

# Save to file
retro_failed_path = "/home/liorkob/M.Sc/thesis/data/5k/gpt/retro_failed_verdicts.csv"
os.makedirs(os.path.dirname(retro_failed_path), exist_ok=True)
failed_df.to_csv(retro_failed_path, index=False, encoding="utf-8-sig")

print(f"🔍 Found {len(failed_df)} failed verdicts.")


### extract indictment facts- no api

In [ ]:
import pandas as pd
import os
import re

# Define start and end patterns based on the 'part' column (for partial matches)
START_PARTS = [
    "עובדותם", "כללי", "כתב האישום", "האישום", "אישום", "רקע", "גזר", "דין", "פסק","מבוא","הרשעת" ,"בעניינו","עבירות","הורשע","עובדות"
]

END_PARTS = [
    "טענות", "עמדת", "תסקיר", "שירות", "מבחן", "דיון", "התסקיר",
    "טיעוני", "הצדדים", "צדדים", "והכרעה", "ראיות"
]

def extract_indictment_facts(df):
    """
    Extracts the 'Indictment Facts' section based on the 'part' column with partial matches.
    Ensures:
    - If start and end are the same, it extends the search.
    - The text **does not** include the content of the end part, only up to it.
    """
    if df.empty or "part" not in df.columns or "text" not in df.columns:
        return "❌ No indictment facts found", None, None

    # Find the first row where 'part' contains a start pattern (case-insensitive, partial match)
    start_row = df[df["part"].str.contains('|'.join(START_PARTS), case=False, na=False, regex=True)]
    if start_row.empty:
        return "❌ No indictment facts found", None, None
    start_idx = start_row.index.min()
    start_part_name = df.loc[start_idx, "part"]

    # Find the first row where 'part' contains an end pattern **after** the start index
    end_row = df[df.index > start_idx][df["part"].str.contains('|'.join(END_PARTS), case=False, na=False, regex=True)]

    # Ensure end is after start and not identical in name
    if not end_row.empty:
        potential_end_idx = end_row.index.min()

        # If the end part is the same as the start part, look further down
        if df.loc[potential_end_idx, "part"] == df.loc[start_idx, "part"]:
            print(f"⚠️ Warning: Start and End have the same name for verdict '{df['verdict'].iloc[0]}'. Searching for next distinct part.")

            # Find the next part that is different from the start part
            extended_end_row = df[df.index > potential_end_idx][df["part"] != df.loc[start_idx, "part"]]

            if not extended_end_row.empty:
                end_idx = extended_end_row.index.min()
            else:
                end_idx = len(df)  # Default to full text if no better match is found
        else:
            end_idx = potential_end_idx  # Use valid end index if found
    else:
        end_idx = len(df)  # Default to full text if no end marker is found

    # Assign extracted part
    end_part_name = df.loc[end_idx, "part"] if end_idx < len(df) else "❌ No end found (used full text)"

    # Extract text **only until** the end part, excluding it
    extracted_text = "\n".join(df.loc[start_idx:end_idx-1, "text"].dropna().astype(str))  # Exclude the last part

    return extracted_text.strip() if extracted_text else "❌ No indictment facts found", start_part_name, end_part_name

# Tracking statistics
total_files = 0
successful_extractions = 0
failed_extractions = 0
failed_verdicts = []
extracted_results = []
for year in [2018,2019,2020]:
    csv_directory = f"/home/liorkob/thesis/lcp/data/docx_csv_{year}"  # Change this to your actual directory

    # Iterate through all CSV files in the directory
    for filename in os.listdir(csv_directory):
        if filename.endswith(".csv"):
            total_files += 1
            file_path = os.path.join(csv_directory, filename)
            
            # Load CSV file
            df = pd.read_csv(file_path)

            # Ensure necessary columns exist
            if 'verdict' not in df.columns or 'text' not in df.columns or 'part' not in df.columns:
                print(f"Skipping {filename}, missing required columns.")
                continue

            # Extract indictment facts based on 'part'
            extracted_facts, start_part, end_part = extract_indictment_facts(df)

            # Track statistics
            if extracted_facts == "❌ No indictment facts found":
                failed_extractions += 1
                failed_verdicts.append({
                    "verdict": df["verdict"].iloc[0],
                    "all_parts": "; ".join(df["part"].dropna().astype(str).unique())  # Store all parts for debugging
                })
                print(f"\n❌ **Failed Extraction for Verdict: {df['verdict'].iloc[0]}**")
                print(f"📌 Available Parts: {failed_verdicts[-1]['all_parts']}\n")
            else:
                successful_extractions += 1

            # Store results
            extracted_results.append({
                "verdict": df["verdict"].iloc[0],
                "extracted_facts": extracted_facts,
                "start_part": start_part if start_part else "❌ Not Found",
                "end_part": end_part if end_part else "❌ Not Found"
            })

# Convert results to DataFrame
final_df = pd.DataFrame(extracted_results)
failed_df = pd.DataFrame(failed_verdicts) if failed_verdicts else pd.DataFrame(columns=["verdict", "all_parts"])

# Save results
final_df.to_csv("processed_verdicts.csv", index=False, encoding="utf-8-sig")
failed_df.to_csv("failed_verdicts.csv", index=False, encoding="utf-8-sig")

# Display statistics
stats = {
    "Total CSV Files Processed": total_files,
    "Successful Extractions": successful_extractions,
    "Failed Extractions": failed_extractions
}

# Print statistics
print("\n=== Statistics ===")
print(pd.DataFrame([stats]))

# Show failed verdicts (if any)
if not failed_df.empty:
    print("\n=== Sample of Failed Verdicts ===")
    print(failed_df.head())  # Print first few rows for review

# Show extracted results with start and end parts
print("\n=== Sample of Successful Extractions (Start & End Parts) ===")
print(final_df[["verdict", "start_part", "end_part"]].head())  # Print first few rows

print("\n✅ Process complete. Results saved as 'processed_verdicts.csv' and 'failed_verdicts.csv'")
